# Riffle Training Pipeline
This notebook handles the SFT and GRPO training for the Chordonomicon project.

### Cell 1 — Setup (run once per session)

In [ ]:
!pip install -q "sentence-transformers==3.4.1" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install --upgrade transformers

  Using cached huggingface_hub-1.14.0-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 30.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.4.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.8.1 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incom

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os, subprocess
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

# System dependencies
!apt-get install -y fluidsynth

is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
_vllm = 'vllm==0.9.2' if is_t4 else 'vllm==0.15.1'
_triton = 'triton==3.2.0' if is_t4 else 'triton'

!pip install uv
!uv pip install -qqq --upgrade {_vllm} bitsandbytes xformers unsloth
!uv pip install -qqq {_triton}
!uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install music21 midiutil midi2audio pydub datasets --quiet

from google.colab import drive
drive.mount('/content/drive')
import sys
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

if not os.path.exists("/content/riffle"):
    !git clone --branch dev https://github.com/ronaldleung1/riffle.git
else:
    !cd /content/riffle && git pull origin dev

sys.path.insert(0, "/content/riffle")
os.chdir("/content/riffle")
print("Ready.")

In [ ]:
# Pull from repo
!git pull origin dev

remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 14 (delta 12), reused 14 (delta 12), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 4.79 KiB | 982.00 KiB/s, done.
From https://github.com/ronaldleung1/riffle
 * branch            dev        -> FETCH_HEAD
   9a596ff..f10c075  dev        -> origin/dev
Updating 9a596ff..f10c075
Fast-forward
 PROGRESS.md                                |  12 +++-
 chord_rewards.py                           |  68 ++++++++++++--------
 grpo_train.py                              |  12 ++--
 scripts/analyze_section_jaccard_dataset.py | 100 +++++++++++++++++------------
 sft_train.py                               |   5 +-
 tests/test_generation_sectional.py         |   4 +-
 tests/test_sectional_reward.py             |  46 ++++++++-----
 7 files changed, 154 insertions(+), 93 deletions(-)


### Cell 2 — SFT smoke test (~3 min)
Confirm loss is decreasing in the logs. If flat or NaN, do not proceed to full run.

In [ ]:
!python sft_train.py \
    --train data/chordonomicon_train.jsonl \
    --val   data/chordonomicon_val.jsonl \
    --out   /content/drive/MyDrive/riffle_checkpoints/sft-smoke \
    --max-train-samples 500 \
    --max-steps 50 \
    --logging-steps 10

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-05-13 07:08:06.795452: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-13 07:08:06.866422: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model: unsloth/Qwen3-1.7B
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2

### Cell 3 — Full SFT (~45–90 min on T4)

In [ ]:
!python sft_train.py \
    --train data/chordonomicon_train.jsonl \
    --val   data/chordonomicon_val.jsonl \
    --out   /content/drive/MyDrive/riffle_checkpoints/sft \
    --epochs 1 \
    --max-train-samples 12000 \
    --bf16

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-05-13 07:13:47.147031: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-13 07:13:47.218414: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model: unsloth/Qwen3-1.7B
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2

(Killed at first checkpoint - step 500)

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen3-1.7B",
    dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-1.7B")

model = PeftModel.from_pretrained(base, "/content/drive/MyDrive/riffle_checkpoints/sft/checkpoint-500")
merged = model.merge_and_unload()

merged.save_pretrained("/content/drive/MyDrive/riffle_checkpoints/sft")
tokenizer.save_pretrained("/content/drive/MyDrive/riffle_checkpoints/sft")

`torch_dtype` is deprecated! Use `dtype` instead!


('/content/drive/MyDrive/riffle_checkpoints/sft/tokenizer_config.json',
 '/content/drive/MyDrive/riffle_checkpoints/sft/special_tokens_map.json',
 '/content/drive/MyDrive/riffle_checkpoints/sft/chat_template.jinja',
 '/content/drive/MyDrive/riffle_checkpoints/sft/vocab.json',
 '/content/drive/MyDrive/riffle_checkpoints/sft/merges.txt',
 '/content/drive/MyDrive/riffle_checkpoints/sft/added_tokens.json',
 '/content/drive/MyDrive/riffle_checkpoints/sft/tokenizer.json')

### Cell 4 — GRPO smoke test (~5 min)
Check that reward in logs is non-zero and varies.

In [ ]:
!python grpo_train.py \
    --train data/chordonomicon_train.jsonl \
    --sft-checkpoint /content/drive/MyDrive/riffle_checkpoints/sft \
    --out   /content/drive/MyDrive/riffle_checkpoints/grpo-smoke \
    --max-train-samples 100 \
    --max-steps 20 \
    --logging-steps 5 \
    --bf16

Traceback (most recent call last):
  File "/content/riffle/grpo_train.py", line 28, in <module>
    from chord_rewards import validate_sectional
  File "/content/riffle/chord_rewards.py", line 29, in <module>
    from music21 import harmony, scale as m21scale
  File "/usr/local/lib/python3.12/dist-packages/music21/__init__.py", line 197, in <module>
    from music21.test.testRunner import mainTest  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/music21/test/__init__.py", line 3, in <module>
    from music21.test import test_base as base
  File "/usr/local/lib/python3.12/dist-packages/music21/test/test_base.py", line 22, in <module>
    from music21 import converter
KeyboardInterrupt
^C


### Cell 5 — Full GRPO (~1–2 hrs on T4)

In [ ]:
!python grpo_train.py \
    --train data/chordonomicon_train.jsonl \
    --sft-checkpoint /content/drive/MyDrive/riffle_checkpoints/sft \
    --out   /content/drive/MyDrive/riffle_checkpoints/grpo \
    --max-steps 500 \
    --bf16

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-05-13 13:44:05.499035: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-13 13:44:05.570101: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading SFT checkpoint: /content/drive/MyDrive/riffle_checkpoints/sft
INFO 05-13 13:44:39 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.15.

### Cell 6 — Three-way eval

In [ ]:
!python eval.py \
    --base-model unsloth/Qwen3-1.7B \
    --sft  /content/drive/MyDrive/riffle_checkpoints/sft \
    --grpo /content/drive/MyDrive/riffle_checkpoints/grpo \
    --samples 5 \
    --out-dir /content/drive/MyDrive/riffle_eval

[base] jazz / intro-verse-chorus-verse-chorus-outro / sample 1
Loading unsloth/Qwen3-1.7B ...
2026-05-13 16:27:35.078733: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.7.0+cu126).
/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
Model loaded.
[base] jazz / intro-verse-chorus-verse-chorus-outro / sample 2
[base] jazz / intro-verse-chorus-verse-chorus-outro / sample 3
[base] jazz / intro-verse-chorus-verse-chorus-outro / sample 4
[base] jazz / intro-verse-chorus-v